# Code mẫu — Fine-tuning

Khi fine-tune, phải nói rõ ba điều: tham số nào được mở cho học, optimizer cập nhật những phần nào, và checkpoint cuối được chọn theo tiêu chí gì. Ta bắt đầu bằng một bước tính tay, rồi thử mở block cuối của một encoder đã học sẵn, và kết thúc bằng một cái bẫy của BatchNorm.

## Ví dụ 1: Mở tham số mà quên đưa vào optimizer

Một encoder nhân đầu vào với `w = 3`, rồi lớp đầu ra nhân tiếp với `a = 1`. Với đầu vào 2 và nhãn 8, dự đoán là 6, và loss bình phương là 4. Gradient theo lớp đầu ra là $2\cdot(6-8)\cdot6=-24$. Gradient theo encoder là $2\cdot(6-8)\cdot1\cdot2=-8$. Ta cho hai phần hai learning rate khác nhau: encoder học chậm hơn để không làm hỏng những gì đã học.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
w = nn.Parameter(torch.tensor(3.0))
a = nn.Parameter(torch.tensor(1.0))
optimizer = torch.optim.SGD([
    {"params": [a], "lr": 0.02},
    {"params": [w], "lr": 0.005},
])
loss = (a * w * 2 - 8)**2
loss.backward()
assert w.grad.item() == -8 and a.grad.item() == -24
print("gradient head/encoder:", a.grad.item(), w.grad.item())
optimizer.step()
print("sau buoc head/encoder:", round(a.item(), 2), round(w.item(), 2))

w2 = nn.Parameter(torch.tensor(3.0))
a2 = nn.Parameter(torch.tensor(1.0))
head_only_optimizer = torch.optim.SGD([a2], lr=0.02)
((a2 * w2 * 2 - 8)**2).backward()
head_only_optimizer.step()
print("quen encoder trong optimizer:", "co gradient =", w2.grad is not None,
      "w =", w2.item())
assert w2.item() == 3

gradient head/encoder: -24.0 -8.0
sau buoc head/encoder: 1.48 3.04
quen encoder trong optimizer: co gradient = True w = 3.0


Lớp đầu ra đổi từ 1 thành $1+0.02\cdot24=1.48$. Encoder chỉ đổi từ 3 thành $3+0.005\cdot8=3.04$, vì learning rate của nó nhỏ hơn.

Phần sau là lỗi hay gặp. Encoder vẫn có gradient, nhưng không nằm trong optimizer, nên `w` đứng yên ở 3. `requires_grad=True` chỉ cho phép tính gradient. Muốn tham số thật sự thay đổi, optimizer phải chứa đúng tham số đó.

## Ví dụ 2: Giữ block đầu, thử mở block cuối

Dữ liệu nguồn gồm 64 điểm trên lưới, gán bốn nhãn theo góc phần tư. Encoder có hai block tuyến tính, `2 → 8 → 4`, được train trên bài nguồn. Bài đích chỉ có 24 điểm train, và cần phân biệt hai phía của đường `x + 0.6y = 0.15`. Toàn bộ là dữ liệu tổng hợp, chạy tại chỗ, không dùng checkpoint công bố nào.

In [ ]:
import copy
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(43)
axis = torch.tensor([-1., -.7, -.4, -.1, .1, .4, .7, 1.])
source_X = torch.cartesian_prod(axis, axis)
source_y = (2 * (source_X[:, 0] > 0) + (source_X[:, 1] > 0)).long()
source_encoder = nn.Sequential(nn.Linear(2, 8), nn.Tanh(),
                               nn.Linear(8, 4), nn.Tanh())
source_model = nn.Sequential(source_encoder, nn.Linear(4, 4))
source_model.train()
source_optimizer = torch.optim.Adam(source_model.parameters(), lr=0.04)
for _ in range(120):
    source_optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(source_model(source_X), source_y)
    loss.backward()
    source_optimizer.step()
source_encoder.zero_grad(set_to_none=True)
source_model.eval()

generator = torch.Generator().manual_seed(430)
target_X = torch.rand((104, 2), generator=generator) * 2 - 1
target_y = (target_X[:, 0] + 0.6 * target_X[:, 1] > 0.15).long()
train_X, valid_X = target_X[:24], target_X[24:]
train_y, valid_y = target_y[:24], target_y[24:]
initial_head = nn.Linear(4, 2)
with torch.no_grad():
    source_correct = int((source_model(source_X).argmax(1) == source_y).sum())
print("nguon dung:", source_correct, "/ 64")
print("target train/valid:", len(train_y), len(valid_y))

nguon dung: 64 / 64
target train/valid: 24 80


Encoder học đúng hết 64 điểm nguồn. Đây chỉ là điểm trên chính dữ liệu đã học, chưa phải điểm trên dữ liệu nguồn giữ riêng. Lớp đầu ra bốn lớp của bài nguồn bị bỏ đi. Bài đích dùng một lớp đầu ra mới hai lớp, khởi tạo một lần và dùng chung cho cả hai cấu hình sắp so.

Hai cấu hình dưới đây cùng học trên 24 điểm, cùng chấm trên 80 điểm validation, và cùng xuất phát từ một encoder và một lớp đầu ra. Cấu hình thứ nhất khóa toàn bộ encoder, chỉ học lớp đầu ra. Cấu hình thứ hai mở thêm block cuối của encoder, với learning rate nhỏ hơn mười lần. Mỗi cấu hình giữ lại checkpoint có validation loss thấp nhất.

In [ ]:
def fit_transfer(open_last):
    encoder = copy.deepcopy(source_encoder)
    head = copy.deepcopy(initial_head)
    encoder.requires_grad_(False)
    if open_last:
        encoder[2].requires_grad_(True)
    model = nn.Sequential(encoder, head)
    before = copy.deepcopy(encoder.state_dict())
    groups = [{"params": head.parameters(), "lr": 0.04}]
    if open_last:
        groups.append({"params": encoder[2].parameters(), "lr": 0.004})
    optimizer = torch.optim.Adam(groups)
    best_loss = float("inf")
    best_state, best_epoch = None, None
    for epoch in range(1, 101):
        model.train()
        encoder.eval()  # Mạng này không có Dropout hay BatchNorm, nhưng vẫn ghi rõ chế độ của encoder.
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(model(train_X), train_y)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            valid_loss = nn.functional.cross_entropy(model(valid_X), valid_y).item()
        if valid_loss < best_loss:
            best_loss, best_epoch = valid_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    with torch.inference_mode():
        correct = int((model(valid_X).argmax(1) == valid_y).sum())
    first_fixed = all(torch.equal(before[name], encoder.state_dict()[name])
                      for name in ["0.weight", "0.bias"])
    last_changed = any(not torch.equal(before[name], encoder.state_dict()[name])
                       for name in ["2.weight", "2.bias"])
    assert first_fixed and last_changed == open_last
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return best_epoch, best_loss, correct, count, first_fixed, last_changed

for opened in [False, True]:
    epoch, loss, correct, count, first_fixed, last_changed = fit_transfer(opened)
    print("mo block cuoi:", opened, "trainable:", count, "epoch:", epoch)
    print("valid loss:", round(loss, 4), "dung:", correct, "/ 80")
    print("block dau giu nguyen:", first_fixed, "block cuoi doi:", last_changed)

mo block cuoi: False trainable: 10 epoch: 57
valid loss: 0.2585 dung: 72 / 80
block dau giu nguyen: True block cuoi doi: False
mo block cuoi: True trainable: 46 epoch: 100
valid loss: 0.1231 dung: 77 / 80
block dau giu nguyen: True block cuoi doi: True


Khi chỉ học lớp đầu ra, có 10 tham số được học, và model đúng 72 trên 80 điểm validation. Mở thêm block cuối, số tham số học tăng lên 46, và số điểm đúng tăng lên 77. Hai dòng kiểm trọng số xác nhận block đầu luôn đứng yên, còn block cuối chỉ đổi khi được mở.

Để ý cột epoch. Cấu hình mở block cuối đạt tốt nhất ở epoch 100, tức epoch cuối cùng, nên có thể nó còn tiến bộ nếu học thêm. Kết quả này chỉ đúng cho cách chia dữ liệu tổng hợp đang xét. Mở thêm tham số không tự bảo đảm tốt hơn ở bài khác, nhất là khi dữ liệu đích ít và dễ bị học thuộc.

## Ví dụ 3: Khóa tham số không giữ yên thống kê của BatchNorm

BatchNorm còn giữ một running mean, nằm ngoài các tham số học được. Ta đặt running mean cũ là 2. Batch mới có trung bình 8, và `momentum = 0.25`. Chỉ một lần forward ở chế độ train đã đổi running mean thành $0.75\cdot2+0.25\cdot8=3.5$, dù không có backward hay optimizer nào.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
bn = nn.BatchNorm1d(1, momentum=0.25)
bn.requires_grad_(False)
bn.running_mean.fill_(2)
batch = torch.tensor([[6.], [10.]])
bn.train()
with torch.no_grad():
    bn(batch)
after = bn.running_mean.clone()
assert after.item() == 3.5
bn.eval()
with torch.no_grad():
    bn(batch + 100)
print("sau train forward:", after.item())
print("eval giu mean:", torch.equal(after, bn.running_mean))
print("tham so co gradient:", any(p.grad is not None for p in bn.parameters()))

sau train forward: 3.5
eval giu mean: True
tham so co gradient: False


Running mean đổi thành 3.5 mà không có bước optimizer nào. Chuyển sang `eval()` thì lần forward tiếp theo không đụng tới nó nữa, kể cả với batch lệch tới 100. Có ba công tắc khác nhau. `requires_grad_(False)` khóa tham số học được. `eval()` khóa thống kê của BatchNorm và tắt dropout. `no_grad()` chỉ tắt việc ghi graph. Khi giữ cố định một encoder có BatchNorm, phải kiểm cả các thống kê này trong `state_dict`, chứ không chỉ kiểm weight và bias.